### Imports

In [1]:
import os, warnings
import faiss
import pandas as pd

from llama_index.core import (
    SimpleDirectoryReader, VectorStoreIndex, StorageContext,
    Settings, PromptTemplate, load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore

from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)   
    from ragas.metrics import (
        faithfulness, answer_relevancy, context_precision,
        context_recall, answer_similarity,
    )
from langchain_ollama import ChatOllama, OllamaEmbeddings
import re

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### RAG Pipeline

In [2]:
model = Ollama(model = "llama3.2:3b", temperature = 0.0, request_timeout = 180.0)
embedding = OllamaEmbedding(model_name="nomic-embed-text")
Settings.llm = model
Settings.embed_model = embedding

# Setting up non-volatile Faiss DB
faiss_dir = "./faiss_index"
os.makedirs(faiss_dir, exist_ok=True)

# Reloading the index instead of re-embedding
if os.path.exists(os.path.join(faiss_dir, "docstore.json")):
    vector_store = FaissVectorStore.from_persist_dir(faiss_dir)
    storage_context = StorageContext.from_defaults(vector_store=vector_store, persist_dir=faiss_dir)
    index = load_index_from_storage(storage_context)
else:
    documents = SimpleDirectoryReader(input_dir="./data_sample").load_data()
    nodes = SentenceSplitter(chunk_size=512, chunk_overlap=50).get_nodes_from_documents(documents)

    dim = len(embedding.get_text_embedding("test"))          
    vector_store = FaissVectorStore(faiss_index=faiss.IndexFlatL2(dim))   
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)
    index.storage_context.persist(persist_dir=faiss_dir)

qa_prompt = PromptTemplate(
    "Context information:\n{context_str}\n\n"
    "Answer the question using ONLY the context above. Be complete, and copy numbers, "
    "ranges and units exactly as written. If several chemistries/items are in the context, "
    "compare them. If the answer is not in the context, say \"I don't know\".\n"
    "Question: {query_str}\nAnswer: "
)      

query_engine = index.as_query_engine(similarity_top_k=5, text_qa_template=qa_prompt)

#### Questions, Contexts & Ground truth

In [3]:
questions = [
    "What is the optimal tilt angle for a site at 40 degrees north latitude?",
    "How often should solar panels be cleaned and what should be used?",
    "What is the maximum pressure washer pressure allowed when cleaning solar panels?",
    "How does shading affect solar panel output and how should it be assessed?",
    "What roof load capacity is required for residential solar installations?",
    "How much energy should a 6 kW system produce per day at 4.2 peak sun hours in a moderate climate?",
    "How much does panel output drop with temperature?",
    "What is the minimum DC cable cross-section for runs under 20 metres, and what are the inverter output cable requirements?",
    "What does fault code F-003 mean and what should be done?",
    "What temperature range can lithium iron phosphate batteries operate in?",
    "Which battery type has the lowest initial cost but the highest lifetime cost, and why?",
    "How does the energy density of NMC batteries compare to other chemistries?",
    "What does the BMS monitor and control in a lithium battery system?",
    "What is the cycle life of LiFePO4 cells at 60% depth of discharge?",
    "How much energy per day does a 10 kWh battery deliver at 60% DOD and how long will it last?",
    "What safety requirements apply to an outdoor residential battery installation?",
    "How is a typical 10 kWh LiFePO4 system configured?"
]


ground_truths = [
    "The optimal tilt angle equals the site's latitude, so a site at 40 degrees north should have panels tilted at 40 degrees from horizontal.",
    "Panels should be cleaned quarterly with a soft brush and deionised water. Abrasive cleaners must not be used.",
    "Pressure washers must not be used above 40 bar, because higher pressure can crack the tempered glass cover.",
    "Even partial shading on a single cell can reduce the output of an entire string by up to 80%. A shading study using a solace sensor or PV*SOL software must be done at 9 AM, noon and 3 PM across equinox and solstice dates.",
    "Residential roofs must support a dead load of at least 30 kg per square meter for the mounting hardware and panels, plus a live load of 1.5 kN per square meter in snow regions. A licensed structural engineer must sign off before mounting.",
    "Before losses it generates 25.2 kWh per day (6,000 W x 4.2 h). After 12-20% system losses, a 6 kW system in a moderate climate should realistically produce 20 to 22 kWh per day.",
    "Temperature derating is approximately 0.35% per degree Celsius above 25 degrees ambient.",
    "DC cabling for runs under 20 metres must be at least 6 mm2. The inverter output cable must be at least 10 mm2 and fused at 80% of the inverter rated current.",
    "F-003 means inverter over-temperature. Ensure ventilation clearance of at least 15 cm on all sides.",
    "LiFePO4 batteries operate from 0 to 45 degrees Celsius.",
    "Lead-acid (AGM/Gel) has the lowest initial cost but the highest lifetime cost because of the need for replacement.",
    "NMC has an energy density of 150-220 Wh/kg, higher than LiFePO4 (90-160 Wh/kg) and lead-acid (30-40 Wh/kg).",
    "The BMS monitors and controls cell voltage (balancing cells within 50 mV), temperature (charging halts above 45 C or below 0 C), charge and discharge current limits, and state of charge estimation.",
    "LiFePO4 cells deliver 3,000+ cycles at 60% depth of discharge.",
    "It delivers about 6 kWh per day and lasts around 3,000 days (about 8 years) before capacity drops below 80% of rated.",
    "The enclosure must be IP65 or higher, installed on a non-combustible surface, with a Class D fire extinguisher within 3 metres and a clearly labelled DC disconnect within 2 metres. Wiring must be rated for max continuous current plus 25%.",
    "It uses 16 LiFePO4 cells (3.2 V nominal, 100 Ah each, matched within 1%) in series for 51.2 V, with a 16S BMS rated 100 A continuous and 200 A peak, and a 5 kW hybrid inverter/charger."
]


gold_contexts = [
    "Tilt angle: The optimal tilt angle equals the latitude of the installation site. For a site at 40 degrees north latitude, the panel should be tilted at 40 degrees from horizontal. A seasonal adjustment of plus or minus 15 degrees can increase annual output by 3-5%.",
    "Quarterly: Clean panels with a soft brush and deionised water. Never use abrasive cleaners or pressure washers above 40 bar, as these can crack the tempered glass cover.",
    "Never use abrasive cleaners or pressure washers above 40 bar, as these can crack the tempered glass cover.",
    "Shading analysis: Even partial shading on a single cell can reduce the output of an entire string by up to 80%. A shading study using a solace sensor or PV*SOL software must be performed at three times of day (9 AM, noon, 3 PM) across equinox and solstice dates.",
    "Structural load: Residential roofs must support a dead load of at least 30 kg per square meter for the mounting hardware and panels, plus a live load of 1.5 kN per square meter for snow regions. A licensed structural engineer must sign off before mounting.",
    "For a 6 kW system at a site with 4.2 peak sun hours per day, the expected daily generation is 6,000 W x 4.2 h = 25.2 kWh per day, before system losses. Total expected system loss: 12-20%. Therefore a 6 kW system in a moderate climate should realistically produce between 20 and 22 kWh per day.",
    "Temperature derating: approximately 0.35% per degree Celsius above 25 degrees ambient",
    "Run DC cabling (minimum 6 mm2 cross-section for runs under 20 metres) through a waterproof conduit from the panel array to the inverter location. The inverter output cable must be minimum 10 mm2 and fused at 80% of the inverter rated current.",
    "Code F-003: Inverter over-temperature. Ensure ventilation clearance of at least 15 cm on all sides.",
    "Lithium Iron Phosphate (LiFePO4): Operating temperature: 0 to 45 degrees C",
    "Lead-Acid (AGM / Gel): Cost: Lowest initial, highest lifetime (due to replacement)",
    "LiFePO4: Energy density: 90-160 Wh/kg. NMC: Energy density: 150-220 Wh/kg. Lead-Acid (AGM / Gel): Energy density: 30-40 Wh/kg",
    "The BMS is the critical safety and performance component of any lithium battery system. It monitors and controls: Cell voltage: Individual cell balancing to keep all cells within 50 mV of each other. Temperature: Charging must be halted if any cell exceeds 45 degrees C or drops below 0 degrees C. Current: Both charge and discharge current must be limited to the cell datasheet maximum. State of Charge (SOC) estimation.",
    "For LiFePO4 cells: 20% DOD: 8,000+ cycles; 40% DOD: 5,000+ cycles; 60% DOD: 3,000+ cycles; 80% DOD: 2,000-3,000 cycles; 100% DOD: 800-1,200 cycles (not recommended)",
    "A 10 kWh system used daily at 60% DOD will deliver approximately 6 kWh per day and last around 3,000 days (8 years) before capacity drops below 80% of rated.",
    "The battery enclosure must be IP65 rated or higher for outdoor installation. The battery must be installed on a non-combustible surface. A Class D fire extinguisher must be available within 3 metres. The DC disconnect switch must be clearly labelled and located within 2 metres of the battery. All wiring must be rated for the maximum continuous current plus 25% for fault conditions.",
    "16 x LiFePO4 cells (3.2V nominal each) in series = 51.2V. BMS: 16S, 100A continuous, 200A peak. Cells: 100 Ah each, matched within 1% capacity. Inverter/Charger: 5 kW hybrid, 51.2V input, 230V AC output."
]


### Result

In [4]:
results = []

for q, gt in zip(questions, ground_truths):
    r = query_engine.query(q)
    results.append({
        "question": q,
        "answer": str(r.response).strip(),
        "ground_truth": gt,
        "contexts": [n.get_content() for n in r.source_nodes],   
    })
    print("=" * 60, "\nQ:", q, "\nA:", results[-1]["answer"])

Q: What is the optimal tilt angle for a site at 40 degrees north latitude? 
A: The optimal tilt angle for a site at 40 degrees north latitude is 40 degrees from horizontal.
Q: How often should solar panels be cleaned and what should be used? 
A: According to the context, solar panels should be cleaned quarterly. The text states:

"Quarterly: Clean panels with a soft brush and deionised water. Never use abrasive cleaners or pressure washers above 40 bar, as these can crack the tempered glass cover."
Q: What is the maximum pressure washer pressure allowed when cleaning solar panels? 
A: 40 bar.
Q: How does shading affect solar panel output and how should it be assessed? 
A: Shading can reduce the output of an entire string of solar panels by up to 80%. A shading study using a solace sensor or PV*SOL software must be performed at three times of day (9 AM, noon, 3 PM) across equinox and solstice dates.
Q: What roof load capacity is required for residential solar installations? 
A: At least

#### Context-coverage score

In [5]:
# Measuring word overlap percentage between document passage and answer
tok = lambda t: set(re.findall(r"[a-z0-9\.\-]+", t.lower()))
for r, g in zip(results, gold_contexts):
    gold, got = tok(g), tok(" ".join(r["contexts"]))
    print(f"{len(gold & got)/len(gold):.0%}  {r['question']}")

100%  What is the optimal tilt angle for a site at 40 degrees north latitude?
100%  How often should solar panels be cleaned and what should be used?
100%  What is the maximum pressure washer pressure allowed when cleaning solar panels?
100%  How does shading affect solar panel output and how should it be assessed?
100%  What roof load capacity is required for residential solar installations?
100%  How much energy should a 6 kW system produce per day at 4.2 peak sun hours in a moderate climate?
100%  How much does panel output drop with temperature?
97%  What is the minimum DC cable cross-section for runs under 20 metres, and what are the inverter output cable requirements?
100%  What does fault code F-003 mean and what should be done?
100%  What temperature range can lithium iron phosphate batteries operate in?
100%  Which battery type has the lowest initial cost but the highest lifetime cost, and why?
92%  How does the energy density of NMC batteries compare to other chemistries?
98%

### Evaluation

In [ ]:

JUDGE_MODEL = "llama3.1:8b"

judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL, temperature=0, num_ctx=8192))
judge_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))


answer_relevancy.strictness = 1

eval_dataset = EvaluationDataset.from_list([
    {
        "user_input": r["question"],
        "response": r["answer"],
        "retrieved_contexts": r["contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

scores = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity],
    llm=judge_llm,
    embeddings=judge_emb,
    run_config=RunConfig(timeout=600, max_workers=1, max_retries=3),   # local models: go sequential
)

df = scores.to_pandas()
df.to_csv("ragas_scores.csv", index=False)
print(df.select_dtypes("number").mean().round(3))

pd.set_option("display.max_colwidth", 60)
cols = ["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall", "semantic_similarity"]
df[[c for c in cols if c in df.columns]].round(2)

C:\Users\User\AppData\Local\Temp\ipykernel_5484\2396141907.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL, temperature=0, num_ctx=8192))
C:\Users\User\AppData\Local\Temp\ipykernel_5484\2396141907.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))
Evaluating: 100%|██████████| 85/85 [10:43<00:00,  7.57s/it]


faithfulness         0.862
answer_relevancy     0.817
context_precision    0.942
context_recall       0.961
answer_similarity    0.859
dtype: float64


,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the optimal tilt angle for a site at 40 degrees ...,1.00,1.00,1.00,1.00
1,How often should solar panels be cleaned and what should...,1.00,0.97,1.00,1.00
2,What is the maximum pressure washer pressure allowed whe...,1.00,0.55,0.70,1.00
3,How does shading affect solar panel output and how shoul...,0.12,0.68,1.00,1.00
4,What roof load capacity is required for residential sola...,1.00,0.73,1.00,1.00
5,How much energy should a 6 kW system produce per day at ...,1.00,0.97,0.89,0.33
6,How much does panel output drop with temperature?,0.50,0.64,1.00,1.00
7,What is the minimum DC cable cross-section for runs unde...,1.00,0.92,1.00,1.00
8,What does fault code F-003 mean and what should be done?,0.67,0.81,1.00,1.00
9,What temperature range can lithium iron phosphate batter...,1.00,0.95,1.00,1.00
